In [ ]:
import os
from pathlib import Path

import polars as pl

In [ ]:
DATABASE_URI = os.environ["DATABASE_URI"]

In [ ]:
RANDOM_SEED = 42

In [ ]:
DATASETS_PATH = Path(
    "/Users/luis/projets/beta.gouv/qfdmod/quefairedemesobjets/data-platform/notebooks/deduplication/datasets"
)

# Annotation manuelle (Christian)

## A partir de la première expèrience ML

In [ ]:
def create_entity_pairs_from_datasets(datasets_folder_path: Path) -> pl.DataFrame:
    csv_files_to_load = datasets_folder_path.glob("Clusterisation *.csv")

    dfs = []
    for csv_filepath in csv_files_to_load:
        df = (
            pl.read_csv(csv_filepath).lazy().rename({"Good ? ": "Good ?"}, strict=False)
        )
        df_pairs = (
            df.join(df, on="cluster_id")
            .filter(pl.col("Good ?").is_in(["oui", "non"]))
            .with_columns(
                pl.min_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_i"),
                pl.max_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_j"),
                (pl.col("Good ?") == "oui").alias("label"),
            )
            .filter(
                (pl.col("identifiant_unique_i") != pl.col("identifiant_unique_j"))
                & (
                    pl.col("cluster_id")
                    .cum_count()
                    .over(
                        ["cluster_id", "identifiant_unique_i", "identifiant_unique_j"]
                    )
                    == 1
                )
            )
            .select(
                ["identifiant_unique_i", "identifiant_unique_j", "label", "cluster_id"]
            )
        )

        df_cluster_ids = (
            df_pairs.filter(pl.col("label"))
            .group_by("cluster_id")
            .agg(
                pl.col("identifiant_unique_i")
                .list.concat(pl.col("identifiant_unique_j"))
                .list.explode()
                .alias("ids")
            )
            .with_columns(
                pl.col("ids")
                .list.unique()
                .sort()
                .hash(RANDOM_SEED)
                .alias("cluster_id_hash")
            )
        )
        df_pairs_with_cluster_id = df_pairs.join(
            df_cluster_ids, on=["cluster_id"], validate="m:1", how="left"
        ).with_columns(
            pl.when(pl.col("label").not_())
            .then(None)
            .otherwise(pl.col("cluster_id_hash"))
            .alias("cluster_id_hash")
        )  # Keep cluster_id only on pairs that are true

        dfs.append(
            df_pairs_with_cluster_id.select(
                [
                    "identifiant_unique_i",
                    "identifiant_unique_j",
                    "label",
                    pl.col("cluster_id_hash").alias("cluster_id").cast(pl.String),
                ]
            )
        )

    df_pairs_concat = (
        pl.concat(dfs)
        .unique(["identifiant_unique_i", "identifiant_unique_j"])
        .sort(["identifiant_unique_i", "identifiant_unique_j"])
        .collect()
    )

    return df_pairs_concat

In [ ]:
df_manual_labeling = create_entity_pairs_from_datasets(DATASETS_PATH)

In [ ]:
df_manual_labeling

In [ ]:
df_manual_labeling.group_by("label").len()

## A partir du fichier d'annotation manuelle

https://docs.google.com/spreadsheets/d/1BARUTEhmugTIppRmkUlLqRuPkxpV2zH-dIShC2G-Ckw/edit?gid=850605265#gid=850605265

### Faux positifs

In [ ]:
df_false_positives_suggestions = pl.read_csv(
    DATASETS_PATH / "false_positives_suggestions.csv"
)

In [ ]:
df_false_positives_suggestions = df_false_positives_suggestions.with_columns(
    pl.lit(False).alias("label")
)

In [ ]:
df_false_positives_suggestions

In [ ]:
df_false_positives_suggestions.write_database(
    "luis.false_positives_suggestions_tmp", connection=DATABASE_URI
)

In [ ]:
df_false_positives_suggestions_pairs = pl.read_database_uri(
    """
    with suggestions as 
(
select 
	fpst.*,
	ds.suggestion->'changes'->0->'model_params'->>'id' as id_parent,
	jsonb_path_query_array(ds.contexte, '$.fuzzy_details[*].identifiant_unique') as identifiant_unique_list
from
	luis.false_positives_suggestions_tmp fpst
inner join data_suggestion ds on
	fpst.suggestion_id = ds.id 
),
suggestions_explosees as (
select
	s.suggestion_id,
	s_lid.id_acteur
from
	suggestions s,
	 jsonb_array_elements_text(s.identifiant_unique_list) as s_lid(id_acteur)
),
paires_dupliquees as (
select
	s1.suggestion_id,
	least(s1.id_acteur , s2.id_acteur ) as identifiant_unique_i,
	greatest(s1.id_acteur , s2.id_acteur) as identifiant_unique_j,
	row_number() over (partition by least(s1.id_acteur , s2.id_acteur ),
	greatest(s1.id_acteur , s2.id_acteur)) as rn
from
	suggestions_explosees s1
inner join suggestions_explosees s2 on
	s1.suggestion_id = s2.suggestion_id
),
paires_dedupliquees as (
select
	pd.suggestion_id,
	identifiant_unique_i,
	identifiant_unique_j
from
	paires_dupliquees pd
where
	rn = 1
	and identifiant_unique_i <> identifiant_unique_j
),
paires_enfants as (
select
	pd.suggestion_id,
	case 
		when qv.identifiant_unique is null then pd.identifiant_unique_i
		else qv.identifiant_unique
	end as identifiant_unique_i,
	case 
		when qv2.identifiant_unique is null then pd.identifiant_unique_j
		else qv2.identifiant_unique
	end as identifiant_unique_j
from
	paires_dedupliquees pd
left join qfdmo_vueacteur qv on
	pd.identifiant_unique_i = qv.parent_id
left join qfdmo_vueacteur qv2 on
	pd.identifiant_unique_j = qv2.parent_id
where
	identifiant_unique_i <> identifiant_unique_j
),
paires_enfant_dupliques as (
select 
	pe.suggestion_id ,
	least(pe.identifiant_unique_i, pe.identifiant_unique_j ) as identifiant_unique_i,
	greatest(pe.identifiant_unique_i, pe.identifiant_unique_j ) as identifiant_unique_j
from
	paires_enfants pe
)
select 
	identifiant_unique_i,
	identifiant_unique_j
from
	(
	select
		*,
		row_number() over (partition by identifiant_unique_i,
		identifiant_unique_j) as rn
	from
		paires_enfant_dupliques 
)
where
	rn = 1
    """,
    uri=DATABASE_URI,
)

In [ ]:
df_false_positives_suggestions_pairs = (
    df_false_positives_suggestions_pairs.with_columns(pl.lit(False).alias("label"))
)

In [ ]:
df_false_positives_suggestions_pairs.shape

# Depuis la base

## Examples négatifs via changement de parent dont l'ancien parent existe toujours

**Vrais négatifs via changement de parent :** Acteurs qui ne sont pas des parents et dont le parent lié aujourd'hui diffère de celui trouvé dans les suggestions de clustering.

In [ ]:
sql_neg_method_1 = """
with suggestions_avec_creation_de_parent as 
(
select
	ds.id,
	ds.statut,
	jsonb_path_query_array(ds.contexte, '$.fuzzy_details[*].identifiant_unique') as identifiant_unique_list,
	ds.suggestion->'changes'->0->'model_params'->>'id' as id_parent,
	ds.suggestion->'changes'->0->>'reason' as suggestion_reason,
	ds.modifie_le
from
	data_suggestion ds
inner join data_suggestioncohorte ds2 
on
	ds.suggestion_cohorte_id = ds2.id
	and ds2.type_action = 'CLUSTERING'
where
	ds.contexte->'fuzzy_details' is not null
	and ds.suggestion->'changes'->0->>'reason' in ('1️⃣ 1 seul parent existant → à garder','➕ Pas de parent → à créer')
	and ds.statut='SUCCES'
),
suggestions_explosees as (
select
	s.id,
	s.id_parent,
	s.suggestion_reason,
	s.modifie_le,
	s_lid.id_acteur
from
	suggestions_avec_creation_de_parent s,
	 jsonb_array_elements_text(s.identifiant_unique_list) as s_lid(id_acteur)
where id_acteur::text<>s.id_parent
),
suggestions_avec_parent_existant as (
SELECT 
	se.*
from suggestions_explosees se
inner join qfdmo_vueacteur qv on se.id_parent=qv.identifiant_unique
),
suggestions_vrais_negatifs as (
select
	 qv.identifiant_unique,
	 qv."uuid",
	 se.id_parent as suggestion_parent_id,
	 se.id as suggestion_id,
	 se.suggestion_reason
from
	qfdmo_vueacteur qv
inner join suggestions_avec_parent_existant se on qv.identifiant_unique = se.id_acteur and qv.parent_id!=se.id_parent
where
	not qv.est_parent
	and qv.statut <> 'SUPPRIME'
	and qv.modifie_le <= now() - interval '1 day'
)
select
	identifiant_unique as identifiant_unique_i,
	suggestion_parent_id as identifiant_unique_j
from suggestions_vrais_negatifs
union all
select
	svn.identifiant_unique as identifiant_unique_i,
	qv.identifiant_unique as identifiant_unique_j
from suggestions_vrais_negatifs svn
inner join 	qfdmo_vueacteur qv
on svn.suggestion_parent_id=qv.parent_id and svn.identifiant_unique!=qv.identifiant_unique
"""

In [ ]:
df_negatives_via_parent_change = pl.read_database_uri(
    query=sql_neg_method_1, uri=DATABASE_URI
).with_columns(
    pl.lit(False).alias("label"), pl.lit(None).alias("cluster_id").cast(pl.String)
)

In [ ]:
df_negatives_via_parent_change

# Par échantillonage aléatoire

**Vrai négatifs :** le but est de créer des paires d'acteurs qui ne sont pas reliées entre elles.
Contraintes :

- Pas de parent en commun
- Pas de relation parent<->enfant
- Pas la même source
- Distance géographique > 10km OU pas le même département

In [ ]:
sql_neg_method_2 = """
WITH
  sampled AS (
    SELECT
      identifiant_unique,
      parent_id,
      source_id,
      "location",
      code_postal,
      acteur_type_id,
      row_number() OVER (
        ORDER BY
          MD5(identifiant_unique::TEXT || '42')
      ) AS rn -- Seed pour le reproductibilité
    FROM
      qfdmo_vueacteur qv
    WHERE
      qv.statut <> 'SUPPRIME'
  ),
  sampled_filtered AS (
    SELECT
      *
    FROM
      sampled
    WHERE
      rn <= 100000 -- Permet la création d'environ 50k paires
  ),
  randomized_pairs AS (
    SELECT
      identifiant_unique,
      parent_id,
      source_id,
      "location",
      code_postal,
      acteur_type_id,
      row_number() OVER (
        ORDER BY
          MD5(identifiant_unique::TEXT || '42')
      ) AS rn -- Mélange aléatoire REPRODUCTIBLE de l'échantillon
    FROM
      sampled_filtered
  )
  -- Création des paires
SELECT
  MIN(a.identifiant_unique) AS identifiant_unique_i,
  MAX(b.identifiant_unique) AS identifiant_unique_j
FROM
  randomized_pairs a
  JOIN randomized_pairs b ON FLOOR((a.rn - 1) / 2) = FLOOR((b.rn - 1) / 2) -- Groupe en paire en utilisant le ~row_number/2 comme clé
  AND a.identifiant_unique < b.identifiant_unique -- S'assure de ne pas avoir de paires avec les mêmes ids dans un ordre différent
  AND a.identifiant_unique <> coalesce(b.parent_id, '') -- acteur_j n'est pas un parent de acteur_i
  AND b.identifiant_unique <> coalesce(a.parent_id, '') -- acteur_i n'est pas un parent de acteur_j
  AND coalesce(a.parent_id, 'N/A') != coalesce(b.parent_id, 'N/A2') -- les deux acteurs n'ont pas le même parent
  AND coalesce(a.source_id, -1) != coalesce(b.source_id, -2) -- les deux acteurs n'ont pas la même source
  AND coalesce(a.acteur_type_id, -1) != coalesce(b.acteur_type_id, -2) -- les deux acteurs ne sont pas du même type
  AND a.acteur_type_id!=10 AND b.acteur_type_id!=10 -- Exclusion des PAV publics
  AND (
    ST_DistanceSpheroid (a."location"::geometry, b."location"::geometry) >= 10000
    OR left(a.code_postal, 2) != left(b.code_postal, 2)
  ) -- plus de 10km entre les deux ou département différent
GROUP BY
  FLOOR((a.rn - 1) / 2)
"""

In [ ]:
df_negatives_random = pl.read_database_uri(
    query=sql_neg_method_2, uri=DATABASE_URI
).with_columns(
    pl.lit(False).alias("label"), pl.lit(None).alias("cluster_id").cast(pl.String)
)

In [ ]:
df_negatives_random

# Vrais positifs

In [ ]:
sql_pos = """
with pool as 
(
select
	qv.identifiant_unique,
	qv.parent_id,
    qv.acteur_type_id,
    qv.source_id,
	case
		when qv.est_parent then qv.identifiant_unique
		else qv.parent_id
	end as cleaned_parent_id -- Le but est de permettre que des parents soient dans le pool (normalement ils n'ont pas de parent_id)
from
	qfdmo_vueacteur qv),
paires_dupliquees as (
select
	least(p.identifiant_unique, p2.identifiant_unique) as identifiant_unique_i,
	greatest(p.identifiant_unique, p2.identifiant_unique) as identifiant_unique_j,
    p.cleaned_parent_id
from
	pool p
inner join pool p2 on
	p.cleaned_parent_id = p2.cleaned_parent_id
	and p.identifiant_unique != p2.identifiant_unique
where 
  coalesce(p.source_id, -1) != coalesce(p2.source_id, -2) -- les deux acteurs n'ont pas la même source
  AND coalesce(p.acteur_type_id, -1) != coalesce(p2.acteur_type_id, -2) -- les deux acteurs ne sont pas du même type
  AND p.acteur_type_id!=10 AND p2.acteur_type_id!=10 -- Exclusion des PAV publics
)
select
	identifiant_unique_i,
	identifiant_unique_j,
    cleaned_parent_id
from
	(
	select
		*,
		row_number() over (partition by identifiant_unique_i,
		identifiant_unique_j) as rn
	from
		paires_dupliquees
)
where
	rn = 1
"""

In [ ]:
df_positives = (
    pl.read_database_uri(query=sql_pos, uri=DATABASE_URI)
    .with_columns(
        pl.lit(True).alias("label"), pl.col("cleaned_parent_id").alias("cluster_id")
    )
    .select(["identifiant_unique_i", "identifiant_unique_j", "label", "cluster_id"])
)

In [ ]:
df_positives

In [ ]:
df_positives.group_by("cluster_id").len().sort("len", descending=True)

In [ ]:
df_positives.group_by("cluster_id").len().mean()

# Constitution d'un dataset équilibré

Même nombre d'examples positifs et négatifs. Pour commencer 1000 de chaque en privilégiant les annotations manuelles en en dernier lieu les paires issues d'un échantillonage aléatoire

In [ ]:
df_positives.sort(pl.col("cluster_id").shuffle())

In [ ]:
df_pairs = pl.concat(
    [
        df_manual_labeling,
        df_negatives_via_parent_change,
        df_negatives_random.sample(
            n=(
                1000
                - len(df_manual_labeling.filter(pl.col("label").not_()))
                - len(df_negatives_via_parent_change)
            ),
            seed=42,
        ),
        df_positives.filter(
            pl.col("cluster_id").is_in(
                df_positives.select(pl.col("cluster_id").unique())
                .sample(
                    n=(1000 - len(df_manual_labeling.filter(pl.col("label")))) // 3.2,
                    seed=42,
                )
                .get_column("cluster_id")
                .to_list()
            )
        ),
    ]
)

In [ ]:
df_pairs

In [ ]:
df_pairs.group_by("label").len()

# Sauvegarde

In [ ]:
df_pairs.write_csv(DATASETS_PATH / "dataset_20260709.csv")